<a href="https://colab.research.google.com/github/andluizsouza/unicamp-llm-agents/blob/main/modules/03_agentes_inteligentes/hands_on_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade Avaliativa — Deep Agent de Planejamento de Viagem

Esta avaliação é um notebook único, com **seis tarefas encadeadas**. Cada tarefa constrói uma peça de um
*deep agent* que planeja uma viagem, e as peças se combinam na última tarefa.

* Professor: Julio Cesar dos Reis <a href="mailto:dosreis@unicamp.br">(dosreis@unicamp.br)</a>
* Monitor: Renan dos Santos Morais <a href="mailto:r299211@dac.unicamp.br">(r299211@dac.unicamp.br)</a>

Você vai implementar:

- as **tools** locais de cálculo;
- a **integração com um servidor MCP** de clima e geolocalização (cliente, não o servidor);
- o **sistema de arquivos**, a **skill** e o **coordenador** com lista de tarefas;
- dois **subagentes**, cada um com uma técnica de planejamento diferente;
- a **integração** de tudo, com traço de execução.

Ao final, há uma seção de análise crítica, que também será avaliada (assim como uma tarefa de implementação).


## 0.1 Como esta avaliação funciona

**Encadeamento.** As tarefas dependem umas das outras: a Tarefa 3 usa o estado definido na Tarefa 2, os
subagentes das Tarefas 4 e 5 são delegados pelo coordenador da Tarefa 3, e a Tarefa 6 integra tudo. Resolva
na ordem.

**O que é avaliado.** Não há dataset e não se avalia a qualidade da viagem produzida. Avalia-se se o
**mecanismo** funciona: as funções devolvem o esperado, o subagente roda em contexto isolado, o arquivo é
gravado e relido, o plano é atualizado. E, na última seção, se você sabe justificar as escolhas.

**Células de verificação.** Depois de cada tarefa há uma célula com `assert` e checagens. Ela imprime `[OK]`
ou `[FALHOU]` para cada critério. Rode antes de seguir.

**Regras.** Consulta livre a notebooks, documentação e material da disciplina. Deixe todas as saídas salvas
no notebook entregue.


## 0.2 Pesos

| Tarefa | O que constrói (objetivo) | Peso |
|---|---|---|
| 1 | Tools locais de cálculo | 10% |
| 2 | Integração com servidor MCP (clima e geolocalização) | 10% |
| 3 | Sistema de arquivos, skill e coordenador com `write_todos` | 20% |
| 4 | Subagente de Orçamento — **ReWOO** | 25% |
| 5 | Subagente de Destino — **Tree of Thoughts** | 25% |
| 6 | Integração  | 10% |


## 0.3 Submissão

* Esta tarefa pode ser realizada individual ou em dupla.
* Apenas **um arquivo por dupla** deve ser submetido.
* Você deve entregar um arquivo do Jupyter Notebook (`.ipynb`), executado do início ao fim, com todas as
  saídas salvas, subdividido conforme as tarefas propostas nesta atividade (Tarefas 1 a 6, mais a análise
  crítica e a especificação escrita da Seção 9). Ver a Seção "8. Checklist de entrega" ao final.
* Junto do notebook, entregue também os arquivos auxiliares gerados durante a resolução — em particular
  `skills/montar-dia-de-roteiro.md` — e o arquivo de configuração do servidor MCP utilizado na Tarefa 2, se
  aplicável. Empacote tudo em um único `.zip`.
* Apenas **um integrante da equipe** deve submeter o arquivo com a solução documentada via Classroom. Nenhum outro canal de entrega será atendido. Respeitar a data e hora limite da entrega conforme descrição no Classroom.
* O arquivo (ou o `.zip`) deve ser nomeado da seguinte forma:

`atividade-deepagent-viagem-<nome_dos_integrantes>.ipynb`

(ou `.zip`, se houver arquivos auxiliares — mantendo o mesmo padrão de nome para o arquivo compactado).


## 0.4 Preenchimento do notebook resposta

* O notebook já vem estruturado por tarefa: enunciado (markdown) → célula(s) de solução → célula de
  verificação. **Não apague, mova ou renomeie** as células de enunciado nem as de verificação.
* Dentro do bloco de cada tarefa, você pode **adicionar quantas células de código ou markdown quiser**
  (funções auxiliares, testes intermediários, explicações do raciocínio, `print`s de depuração etc.). O
  único requisito é que, ao final do bloco da tarefa — antes da célula de verificação correspondente —
  todos os nomes exigidos (funções, classes, variáveis como `coordenador`, `subagente_orcamento`, `T`, ...)
  estejam definidos exatamente com o nome usado nas checagens (`verificar(...)`), pois é isso que a
  verificação importa do namespace do notebook.
* Pode reorganizar livremente as células **dentro do bloco de uma mesma tarefa**, mas não mova células de
  solução para o bloco de outra tarefa — a numeração e o encadeamento dependem da ordem das tarefas.
* Se precisar de uma biblioteca além das já instaladas, adicione-a na célula `%pip install` da Seção 0.7
  (Preparação do ambiente), não em uma célula solta no meio de uma tarefa.
* Células de texto explicando decisões de projeto são bem-vindas, mas não substituem código funcional —
  o que é avaliado é o comportamento verificado pela célula `verificar(...)` de cada tarefa.
* Antes de entregar, rode o notebook do início ao fim em uma sessão limpa (**Kernel/Runtime → Restart and
  Run All**) para garantir que ele funciona na ordem em que será corrigido, e deixe todas as saídas salvas.


## 0.5 O cenário

O cenário é **fixo e igual para todos**. Não altere os valores — as verificações dependem deles.

> **Viagem:** duas pessoas saindo de São Paulo (GRU), de **12 a 19 de outubro de 2026**.
> **Orçamento total:** R$ 12.000,00, incluindo passagens, hospedagem, transporte local e alimentação.
> **Interesses:** gastronomia, museus e natureza urbana.
> **Restrição 1:** um dos viajantes tem mobilidade reduzida — trajetos a pé acima de 2 km e atrações sem
> acessibilidade devem ser evitados.
> **Restrição 2:** no dia **15 de outubro** há um feriado local no destino, com a maior parte dos museus e
> comércio fechados.
> **Destinos candidatos:** Buenos Aires, Santiago e Montevidéu. O agente deve escolher **um**.

Os candidatos são fixos de propósito: isso torna a árvore de decisão da Tarefa 6 comparável entre todos.

## 0.6 Mapa das tarefas

1. Tools locais de cálculo.
2. Integração com um servidor MCP de clima e geolocalização (Open-Meteo / Nominatim).
3. Sistema de arquivos, skill e coordenador.
4. Subagente de Orçamento com ReWOO.
5. Subagente de Destino com Tree of Thoughts.
6. Integração e traço de execução.
7. Análise crítica.
8. Especificação escrita (Plan-and-Execute e Reflection).

> As Seções 7 e 8 (análise crítica e especificação escrita) são referenciadas no checklist de entrega, mas
> ainda não têm células próprias neste notebook — adicione-as antes de aplicar esta avaliação, com as
> perguntas específicas que você quer que os alunos respondam.


---

## 0.7 Preparação do ambiente

**Não modifique as células desta seção.** Apenas execute-as na ordem — elas definem funções auxiliares e
utilitários de verificação usados por todas as tarefas.


In [ ]:
# Descomente se estiver em um ambiente sem as dependências instaladas.
%pip install -U langchain langchain-core langgraph langchain-openai python-dotenv pydantic rich
%pip install -U mcp langchain-mcp-adapters httpx
%pip install -U deepagents


In [ ]:
from __future__ import annotations

import asyncio
import json
import os
from datetime import date, datetime, timedelta
from typing import Annotated, Any, Dict, List, Literal, Optional, TypedDict

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain.tools import tool as tool_decorator
from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)
from langchain_core.tools import InjectedToolCallId

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.graph.state import CompiledStateGraph
from langgraph.prebuilt import InjectedState
from langgraph.types import Command

from IPython.display import Image, display
from rich import print
from rich.markdown import Markdown

In [ ]:
def display_graph(graph: CompiledStateGraph) -> None:
    display(Image(graph.get_graph().draw_mermaid_png()))


def print_json(value: Any) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def print_messages(messages: List[BaseMessage]) -> None:
    for index, message in enumerate(messages):
        print(f"\nMensagem {index} — {message.__class__.__name__}")
        if isinstance(message, AIMessage) and message.tool_calls:
            print("tool_calls:", [c["name"] for c in message.tool_calls])
        else:
            conteudo = str(message.content)
            print(conteudo[:300] + (" [...]" if len(conteudo) > 300 else ""))


def contar_tokens_aproximado(messages: List[BaseMessage]) -> int:
    total = 0
    for m in messages:
        total += len(str(m.content))
        if isinstance(m, AIMessage) and m.tool_calls:
            total += len(json.dumps(m.tool_calls, ensure_ascii=False))
    return total // 4


_RESULTADOS_VERIFICACAO: Dict[str, List[bool]] = {}


def verificar(tarefa: str, criterio: str, condicao: bool, detalhe: str = "") -> None:
    """Registra e imprime o resultado de um critério de verificação."""
    _RESULTADOS_VERIFICACAO.setdefault(tarefa, []).append(bool(condicao))
    marca = "[OK]     " if condicao else "[FALHOU] "
    print(f"{marca}{criterio}" + (f"  ->  {detalhe}" if detalhe and not condicao else ""))


def resumo_verificacoes() -> None:
    print("\n=== RESUMO DAS VERIFICAÇÕES ===")
    for tarefa, resultados in _RESULTADOS_VERIFICACAO.items():
        print(f"{tarefa:34} {sum(resultados)}/{len(resultados)} critérios")

In [ ]:
load_dotenv()
print("Chave OPENAI_API_KEY configurada." if "OPENAI_API_KEY" in os.environ else "Chave não encontrada.")

llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

### Dados do cenário

**Não modifique os dicionários abaixo (`CENARIO`, `CATALOGO_VOOS`, `CATALOGO_HOSPEDAGENS`,
`CUSTOS_DIARIOS_BRL`, `COORDENADAS`).** Estes dados são fornecidos. O catálogo de voos e hospedagens é
**fictício e controlado**, para que a avaliação não dependa de APIs de reserva que mudam ou bloqueiam acesso.
As células de verificação dependem exatamente destes valores.


In [ ]:
# NÃO MODIFIQUE ESTE BLOCO — valores fixos usados pelas verificações de todas as tarefas.
CENARIO = {
    "origem": "São Paulo",
    "origem_iata": "GRU",
    "data_ida": "2026-10-12",
    "data_volta": "2026-10-19",
    "viajantes": 2,
    "orcamento_total_brl": 12000.00,
    "interesses": ["gastronomia", "museus", "natureza urbana"],
    "restricoes": [
        "um viajante com mobilidade reduzida: evitar trajetos a pé acima de 2 km e atrações sem acessibilidade",
        "feriado local em 2026-10-15: museus e comércio majoritariamente fechados",
    ],
    "candidatos": ["Buenos Aires", "Santiago", "Montevidéu"],
}

CATALOGO_VOOS = {
    "Buenos Aires": {"iata": "EZE", "preco_brl": 1850.00, "duracao_h": 2.8, "escalas": 0},
    "Santiago":     {"iata": "SCL", "preco_brl": 2300.00, "duracao_h": 4.2, "escalas": 0},
    "Montevidéu":   {"iata": "MVD", "preco_brl": 2050.00, "duracao_h": 3.1, "escalas": 1},
}

CATALOGO_HOSPEDAGENS = {
    "Buenos Aires": [
        {"nome": "Hotel Recoleta", "diaria_brl": 420.00, "acessivel": True,  "nota": 8.6},
        {"nome": "Hostel Palermo", "diaria_brl": 180.00, "acessivel": False, "nota": 8.1},
    ],
    "Santiago": [
        {"nome": "Hotel Providencia", "diaria_brl": 480.00, "acessivel": True,  "nota": 8.8},
        {"nome": "Apart Bellas Artes", "diaria_brl": 310.00, "acessivel": True, "nota": 8.3},
    ],
    "Montevidéu": [
        {"nome": "Hotel Pocitos", "diaria_brl": 350.00, "acessivel": True,  "nota": 8.4},
        {"nome": "Pousada Ciudad Vieja", "diaria_brl": 240.00, "acessivel": False, "nota": 7.9},
    ],
}

CUSTOS_DIARIOS_BRL = {
    "Buenos Aires": {"alimentacao": 180.00, "transporte_local": 60.00},
    "Santiago":     {"alimentacao": 210.00, "transporte_local": 80.00},
    "Montevidéu":   {"alimentacao": 190.00, "transporte_local": 55.00},
}

COORDENADAS = {
    "Buenos Aires": (-34.6037, -58.3816),
    "Santiago":     (-33.4489, -70.6693),
    "Montevidéu":   (-34.9011, -56.1645),
}

print_json(CENARIO)

# Tarefa 1 — Tools locais de cálculo *(10%)*

As tools desta tarefa são **puras**: recebem argumentos, calculam e devolvem. Nenhuma acessa rede, nenhuma lê
o estado do grafo. É a camada determinística do agente.

Implemente as quatro tools abaixo, com o decorador `@tool_decorator`. Todas devem devolver **string JSON**
com `ensure_ascii=False`, seguindo a convenção dos notebooks da disciplina.

**`calcular_dias(data_inicio, data_fim)`**
Devolve `{"dias": N, "noites": N-1}`. Datas no formato `YYYY-MM-DD`. Se `data_fim` for anterior a
`data_inicio`, devolva `{"erro": "datas_invalidas"}`.

**`somar_orcamento(itens)`**
Recebe uma lista de dicionários com `categoria` e `valor_brl`. Devolve
`{"total_brl": T, "por_categoria": {...}}`, com os valores arredondados em duas casas.

**`tempo_deslocamento(distancia_km, modal)`**
Modais e velocidades médias: `"a_pe"` 4 km/h, `"transporte_publico"` 18 km/h, `"taxi"` 25 km/h. Devolve
`{"minutos": M, "modal": modal}`, com `M` inteiro arredondado. Modal desconhecido devolve
`{"erro": "modal_invalido", "modais_disponiveis": [...]}`.

**`validar_datas_roteiro(roteiro)`**
Recebe uma lista de dicionários com `data` e `atividades`. Devolve
`{"valido": bool, "conflitos": [...]}`. Conta como conflito: data duplicada, data fora do intervalo da
viagem, e dia sem nenhuma atividade.

In [ ]:
# Escreva sua solução aqui.

### Verificação — Tarefa 1

**Não modifique esta célula.** Ela só deve ser executada, para checar sua solução.


In [ ]:
T = "Tarefa 1 — tools locais"

r = json.loads(calcular_dias.invoke({"data_inicio": "2026-10-12", "data_fim": "2026-10-19"}))
verificar(T, "calcular_dias conta 8 dias e 7 noites", r.get("dias") == 8 and r.get("noites") == 7, str(r))

r = json.loads(calcular_dias.invoke({"data_inicio": "2026-10-19", "data_fim": "2026-10-12"}))
verificar(T, "calcular_dias rejeita datas invertidas", r.get("erro") == "datas_invalidas", str(r))

itens = [
    {"categoria": "passagem", "valor_brl": 1850.0},
    {"categoria": "hospedagem", "valor_brl": 2940.0},
    {"categoria": "passagem", "valor_brl": 1850.0},
]
r = json.loads(somar_orcamento.invoke({"itens": itens}))
verificar(T, "somar_orcamento totaliza corretamente", abs(r.get("total_brl", 0) - 6640.0) < 0.01, str(r))
verificar(T, "somar_orcamento agrupa por categoria",
          abs(r.get("por_categoria", {}).get("passagem", 0) - 3700.0) < 0.01, str(r))

r = json.loads(tempo_deslocamento.invoke({"distancia_km": 6, "modal": "transporte_publico"}))
verificar(T, "tempo_deslocamento calcula 6 km de transporte público", r.get("minutos") == 20, str(r))

r = json.loads(tempo_deslocamento.invoke({"distancia_km": 3, "modal": "helicoptero"}))
verificar(T, "tempo_deslocamento rejeita modal inválido", r.get("erro") == "modal_invalido", str(r))

roteiro_ok = [{"data": f"2026-10-{d}", "atividades": ["visita"]} for d in range(12, 20)]
r = json.loads(validar_datas_roteiro.invoke({"roteiro": roteiro_ok}))
verificar(T, "validar_datas_roteiro aceita roteiro válido", r.get("valido") is True, str(r))

roteiro_ruim = [
    {"data": "2026-10-12", "atividades": ["visita"]},
    {"data": "2026-10-12", "atividades": ["outra"]},
    {"data": "2026-11-01", "atividades": ["fora do intervalo"]},
    {"data": "2026-10-14", "atividades": []},
]
r = json.loads(validar_datas_roteiro.invoke({"roteiro": roteiro_ruim}))
verificar(T, "validar_datas_roteiro detecta os três tipos de conflito",
          r.get("valido") is False and len(r.get("conflitos", [])) >= 3, str(r))

# Tarefa 2 — Integração com servidor MCP *(10%)*

Nesta tarefa você **conecta-se, como cliente**, a um servidor MCP que expõe dados reais de clima e
geolocalização. Você não precisa escrever o lado servidor do protocolo — use um servidor MCP público e
gratuito para Open-Meteo/Nominatim (por exemplo, algum dos listados em
[mcpservers.org](https://mcpservers.org) ou o indicado pela disciplina), rodando localmente via `npx` ou
`uvx`, sem chave de acesso.

O servidor deve expor, no mínimo, duas capacidades equivalentes a:

**`geocodificar(local)`** — converte um nome de lugar em coordenadas (Nominatim/OpenStreetMap).

**`prever_clima(latitude, longitude, data_inicio, data_fim)`** — obtém a previsão diária de temperatura e
precipitação (Open-Meteo).

Três requisitos que valem nota, do **lado cliente**:

1. **Conexão e listagem de tools** — conecte-se ao servidor com `MultiServerMCPClient` (transporte `stdio`) e
   liste as tools disponíveis.
2. **Cache em memória do lado cliente**, para que a mesma consulta não vá ao servidor duas vezes.
3. **Tratamento de falha com fallback** — se a chamada ao servidor falhar (rede indisponível, servidor fora
   do ar), use os dados de `COORDENADAS` como alternativa. A verificação aceita o fallback.

> Se preferir demonstrar como um servidor MCP é escrito, isso pode ser feito como exercício **bônus**, fora
> da nota desta tarefa — não é o foco da avaliação.


Configure a conexão com o servidor MCP escolhido (comando, argumentos, transporte `stdio`) e liste as
tools disponíveis com `MultiServerMCPClient`.


In [ ]:
# Escreva sua solução aqui.
# Sugestão: use MultiServerMCPClient de langchain_mcp_adapters.client com transporte "stdio",
# apontando para o comando (ex.: npx/uvx) do servidor MCP escolhido.


Implemente o cache e o fallback do lado cliente ao redor das chamadas às tools do servidor.


In [ ]:
# Escreva sua solução aqui.
# 'tools_mcp' deve conter as tools listadas pelo cliente MCP.


### Verificação — Tarefa 2

**Não modifique esta célula.** Ela só deve ser executada, para checar sua solução.


In [ ]:
T = "Tarefa 2 — integração MCP"

# 'tools_mcp' deve ter sido definida por você na célula do cliente.
nomes = [t.name for t in tools_mcp] if "tools_mcp" in dir() else []
verificar(T, "cliente MCP listou tools de geocodificação e clima",
          any("geo" in n.lower() for n in nomes) and any("clima" in n.lower() or "weather" in n.lower() or "forecast" in n.lower() for n in nomes),
          str(nomes))
verificar(T, "existe mecanismo de cache do lado cliente", "cache" in globals() or "CACHE_MCP" in globals())
verificar(T, "existe fallback com COORDENADAS em caso de falha",
          "COORDENADAS" in str(globals().get("obter_tools_mcp", "")) or True,
          "confira manualmente se o fallback está implementado")


# Tarefa 3 — Sistema de arquivos, skill e coordenador *(20%)*

Três peças aqui: o estado com memória em arquivo, o conhecimento procedimental, e o loop que os consome.

## 3.1 O sistema de arquivos

Use o sistema de arquivos **pronto** da biblioteca `deepagents` (`FilesystemMiddleware` com um `StateBackend`,
por exemplo) em vez de reimplementar `ls`/`read_file`/`write_file` do zero — veja a documentação em
[docs.langchain.com/oss/python/deepagents/overview](https://docs.langchain.com/oss/python/deepagents/overview).

Você só precisa:

- definir **`EstadoViagem`**, um `TypedDict` com os campos próprios do cenário — `messages` (reducer
  `add_messages`), `todos` (lista de tarefas do coordenador) e `cenario` (o dicionário do cenário, sem
  reducer) — que se **compõe** com o `files` fornecido pelo middleware de filesystem;
- configurar o backend (ex.: `StateBackend()`) e confirmar que os tools `ls`, `read_file`, `write_file`
  ficam disponíveis para o coordenador e os subagentes.

> Confira a versão instalada do `deepagents` — a API de middleware/backends é recente e pode variar entre
> versões da biblioteca.

## 3.2 A skill

Escreva o arquivo `skills/montar-dia-de-roteiro.md`, no formato da aula de Skills: *frontmatter* com `name` e
`description`, e corpo com o procedimento.

A descrição precisa dizer **quando a skill se aplica e quando não se aplica** — é por ela que o agente decide
ativar. O corpo deve conter, no mínimo:

- máximo de atividades por dia e folga mínima entre deslocamentos;
- a regra de agrupar atividades por proximidade geográfica;
- como tratar as duas restrições do cenário (mobilidade reduzida e feriado de 15/10);
- o que fazer quando uma atração está fechada.

Instruções procedimentais, em linguagem natural. Sem lógica condicional complexa — isso seria uma tool.

## 3.3 O coordenador

Implemente:

- **`write_todos(todos, tool_call_id)`** — tool que grava a lista de tarefas no estado, devolvendo `Command`.
  Cada item tem `descricao` e `status` (`pendente`, `em_andamento`, `concluida`).
- **`SYSTEM_COORDENADOR`** — o prompt de sistema, carregando o conteúdo da skill e prescrevendo o
  procedimento: planejar antes de agir, gravar resultados longos em arquivo, delegar, marcar concluído.
- **`no_llm_coordenador`**, **`no_tools_coordenador`**, **`rotear_coordenador`** e o grafo compilado em
  **`coordenador`**, com limite explícito de passos.


In [ ]:
# Escreva sua solução aqui — EstadoViagem e a configuração do sistema de arquivos (deepagents).


In [ ]:
# Escreva sua solução aqui — a skill.
# Sugestão: use %%writefile skills/montar-dia-de-roteiro.md em uma célula própria.


In [ ]:
# Escreva sua solução aqui — write_todos, o prompt de sistema e o grafo do coordenador.


### Verificação — Tarefa 3

**Não modifique esta célula.** Ela só deve ser executada, para checar sua solução.


In [ ]:
T = "Tarefa 3 — sistema de arquivos, skill e coordenador"

verificar(T, "EstadoViagem tem os campos esperados",
          {"messages", "todos", "cenario"}.issubset(set(EstadoViagem.__annotations__)),
          str(set(EstadoViagem.__annotations__)))

estado = {"files": {"/destinos.md": "Buenos Aires"}}
verificar(T, "ls lista os arquivos", "/destinos.md" in ls.invoke({"state": estado}))
verificar(T, "read_file lê um arquivo existente",
          read_file.invoke({"caminho": "/destinos.md", "state": estado}) == "Buenos Aires")
verificar(T, "read_file avisa quando o arquivo não existe",
          "erro" in read_file.invoke({"caminho": "/nao_existe.md", "state": estado}).lower())

cmd = write_file.invoke({"name": "write_file", "args": {"caminho": "/x.md", "conteudo": "abc"},
                         "id": "c1", "type": "tool_call"})
verificar(T, "write_file devolve Command", isinstance(cmd, Command))
verificar(T, "write_file grava no campo files", cmd.update.get("files") == {"/x.md": "abc"}, str(cmd.update))

caminho_skill = "skills/montar-dia-de-roteiro.md"
verificar(T, "arquivo da skill foi criado", os.path.exists(caminho_skill))

skill = open(caminho_skill, encoding="utf-8").read() if os.path.exists(caminho_skill) else ""
verificar(T, "skill tem frontmatter com name", "name:" in skill)
verificar(T, "skill tem frontmatter com description", "description:" in skill)
verificar(T, "skill trata a restrição de mobilidade", "mobilidade" in skill.lower())
verificar(T, "skill trata o feriado de 15/10", "feriado" in skill.lower() or "15" in skill)
verificar(T, "skill tem corpo substantivo", len(skill) > 400, f"{len(skill)} caracteres")

cmd = write_todos.invoke({"name": "write_todos",
                          "args": {"todos": [{"descricao": "Escolher destino", "status": "pendente"}]},
                          "id": "c1", "type": "tool_call"})
verificar(T, "write_todos devolve Command", isinstance(cmd, Command))
verificar(T, "write_todos atualiza o campo todos", len(cmd.update.get("todos", [])) == 1, str(cmd.update))

verificar(T, "prompt de sistema carrega o conteúdo da skill",
          any(t in SYSTEM_COORDENADOR.lower() for t in ["mobilidade", "roteiro", "proximidade"]))
verificar(T, "prompt prescreve o uso de write_todos", "write_todos" in SYSTEM_COORDENADOR)
verificar(T, "coordenador é um grafo compilado", isinstance(coordenador, CompiledStateGraph))


# Tarefa 4 — Subagente de Orçamento com ReWOO *(25%)*

Este subagente calcula o orçamento da viagem para um destino dado, e usa **ReWOO**.

ReWOO é a escolha certa aqui porque todas as consultas são **independentes e conhecidas de antemão**:
passagem, hospedagem, alimentação e transporte local não dependem umas das outras. Um único planejamento,
execução com substituição de variáveis, e uma síntese. É o padrão mais barato em número de chamadas de LLM.

## 5.1 O planejador

Um nó que produz, em **uma única chamada de LLM**, a lista completa de passos com variáveis:

```text
Plano: buscar o preço da passagem para o destino
#E1 = consultar_voo[Buenos Aires]
Plano: buscar a hospedagem acessível mais barata
#E2 = consultar_hospedagem[Buenos Aires]
Plano: somar o orçamento com os valores obtidos
#E3 = somar_orcamento[#E1, #E2]
```

Use `with_structured_output` com um modelo Pydantic — não faça parsing de texto livre.

## 5.2 O executor

Percorre os passos na ordem, **substituindo as variáveis** `#En` pelos resultados já obtidos, e chama a tool
correspondente. Sem chamada de LLM aqui.

## 5.3 O solver

Uma chamada final de LLM que sintetiza os resultados em um orçamento comentado, e grava em
`/orcamento.md` pelo `write_file`.

## 5.4 A tool de delegação

**`delegar_orcamento(destino, state, tool_call_id)`** — roda o subagente em contexto próprio e devolve ao
coordenador apenas o resumo, com os arquivos gravados. O histórico interno **não** volta.

Você precisará das tools `consultar_voo` e `consultar_hospedagem`, que devem ler `CATALOGO_VOOS`,
`CATALOGO_HOSPEDAGENS` e `CUSTOS_DIARIOS_BRL`. Implemente-as também.

In [ ]:
# Escreva sua solução aqui — as tools de catálogo.

In [ ]:
# Escreva sua solução aqui — planejador, executor, solver e o grafo do subagente.

In [ ]:
# Escreva sua solução aqui — a tool delegar_orcamento.

### Verificação — Tarefa 4

**Não modifique esta célula.** Ela só deve ser executada, para checar sua solução.


In [ ]:
T = "Tarefa 4 — subagente ReWOO"

r = json.loads(consultar_voo.invoke({"destino": "Buenos Aires"}))
verificar(T, "consultar_voo devolve o preço do catálogo", abs(r.get("preco_brl", 0) - 1850.0) < 0.01, str(r))

r = json.loads(consultar_hospedagem.invoke({"destino": "Santiago"}))
verificar(T, "consultar_hospedagem devolve opções", len(r.get("opcoes", r if isinstance(r, list) else [])) >= 1, str(r))

verificar(T, "subagente de orçamento é um grafo compilado", isinstance(subagente_orcamento, CompiledStateGraph))

nos = set(subagente_orcamento.get_graph().nodes)
verificar(T, "grafo tem os três papéis do ReWOO",
          sum(1 for n in nos if any(p in n.lower() for p in ["plan", "exec", "solv", "work"])) >= 3, str(nos))

resultado = subagente_orcamento.invoke({
    "messages": [HumanMessage(content="Calcule o orçamento para Buenos Aires.")],
    "todos": [], "files": {}, "cenario": CENARIO,
})

chamadas_llm = sum(1 for m in resultado["messages"] if isinstance(m, AIMessage))
verificar(T, "ReWOO usa poucas chamadas de LLM (<= 4)", chamadas_llm <= 4, f"{chamadas_llm} chamadas")
verificar(T, "subagente gravou /orcamento.md", "/orcamento.md" in (resultado.get("files") or {}),
          str(list((resultado.get('files') or {}).keys())))

cmd = delegar_orcamento.invoke({
    "name": "delegar_orcamento",
    "args": {"destino": "Buenos Aires", "state": {"files": {}, "cenario": CENARIO}},
    "id": "c1",
    "type": "tool_call",
})
verificar(T, "delegar_orcamento devolve Command", isinstance(cmd, Command))
verificar(T, "delegação devolve arquivos ao coordenador", "/orcamento.md" in (cmd.update.get("files") or {}))
verificar(T, "delegação devolve UMA mensagem ao coordenador", len(cmd.update.get("messages", [])) == 1,
          "o histórico interno do subagente não deve voltar")

# Tarefa 5 — Subagente de Destino com Tree of Thoughts *(25%)*

Este subagente escolhe **um** dos três destinos candidatos, e usa **Tree of Thoughts**.

ToT é a escolha certa aqui porque o problema é de **seleção entre alternativas**: gerar candidatos, avaliar
cada um e podar os piores. Não há sequência de ações a planejar — há um espaço de opções a explorar.

## 6.1 A árvore

A árvore tem dois níveis e **seis folhas**:

```text
                        raiz
        ┌────────────────┼────────────────┐
   Buenos Aires      Santiago        Montevidéu
     ┌────┴────┐     ┌────┴────┐     ┌────┴────┐
   gastro.  natureza gastro. natureza gastro. natureza
```

Nível 1: os três destinos candidatos do cenário.
Nível 2: para cada destino, **duas abordagens de viagem** — uma com ênfase em gastronomia e museus, outra com
ênfase em natureza urbana.

## 6.2 Os três nós

**Gerador** — produz as seis folhas, cada uma com uma descrição curta da viagem proposta.

**Avaliador** — pontua cada folha de 0 a 10 em quatro critérios, usando saída estruturada:
custo estimado, clima na janela de datas, aderência aos interesses, e **acessibilidade** (a restrição de
mobilidade reduzida). Use as tools MCP da Tarefa 2 para o clima, e as tools de catálogo da Tarefa 5 para o
custo — a avaliação não pode ser puro chute do modelo.

**Podador** — descarta as folhas abaixo da mediana, escolhe a melhor entre as restantes, e grava a decisão
com a justificativa em `/destinos.md`.

## 6.3 A tool de delegação

**`delegar_destino(state, tool_call_id)`** — mesmo padrão da Tarefa 5.

In [ ]:
# Escreva sua solução aqui — os modelos Pydantic da árvore.

In [ ]:
# Escreva sua solução aqui — gerador, avaliador, podador e o grafo do subagente.

In [ ]:
# Escreva sua solução aqui — a tool delegar_destino.

### Verificação — Tarefa 5

**Não modifique esta célula.** Ela só deve ser executada, para checar sua solução.


In [ ]:
T = "Tarefa 5 — subagente ToT"

verificar(T, "subagente de destino é um grafo compilado", isinstance(subagente_destino, CompiledStateGraph))

nos = set(subagente_destino.get_graph().nodes)
verificar(T, "grafo tem os três papéis do ToT",
          sum(1 for n in nos if any(p in n.lower() for p in ["ger", "aval", "pod", "prun", "select"])) >= 3,
          str(nos))

resultado = subagente_destino.invoke({
    "messages": [HumanMessage(content="Escolha o destino da viagem.")],
    "todos": [], "files": {}, "cenario": CENARIO,
})

verificar(T, "subagente gravou /destinos.md", "/destinos.md" in (resultado.get("files") or {}),
          str(list((resultado.get('files') or {}).keys())))

conteudo = (resultado.get("files") or {}).get("/destinos.md", "")
verificar(T, "arquivo cita os três candidatos",
          all(c in conteudo for c in CENARIO["candidatos"]), "as folhas avaliadas devem aparecer")
verificar(T, "arquivo registra um destino escolhido",
          any(c in conteudo for c in CENARIO["candidatos"]) and len(conteudo) > 200,
          f"{len(conteudo)} caracteres")
verificar(T, "avaliação considerou acessibilidade",
          "acess" in conteudo.lower() or "mobilidade" in conteudo.lower())

cmd = delegar_destino.invoke({"name": "delegar_destino",
                              "args": {"state": {"files": {}, "cenario": CENARIO}},
                              "id": "c1", "type": "tool_call"})
verificar(T, "delegar_destino devolve Command", isinstance(cmd, Command))
verificar(T, "delegação devolve UMA mensagem ao coordenador", len(cmd.update.get("messages", [])) == 1)

# Tarefa 6 — Integração *(10%)*

Amarre tudo em um único deep agent e execute a tarefa completa.

O coordenador deve ter acesso a: `write_todos`, `ls`, `read_file`, `write_file`, as tools locais da Tarefa 1,
as tools MCP da Tarefa 2, e as duas tools de delegação. Compile em **`agente_viagem`**.

A tarefa a executar:

> Planeje a viagem descrita no cenário. Escolha o destino, calcule o orçamento e produza um guia final com a
> recomendação, o resumo de custos e as ressalvas sobre as restrições. Grave o guia em `/guia_final.md`.

Depois de executar, imprima o traço em três partes: a **lista de tarefas final**, o **sistema de arquivos**
com o tamanho de cada arquivo, e a **sequência de chamadas de tool**.

In [ ]:
# Escreva sua solução aqui — montagem do agente completo.

In [ ]:
# Escreva sua solução aqui — execução da tarefa e traço.

### Verificação — Tarefa 6

**Não modifique esta célula.** Ela só deve ser executada, para checar sua solução.


In [ ]:
T = "Tarefa 6 — integração"

verificar(T, "agente_viagem é um grafo compilado", isinstance(agente_viagem, CompiledStateGraph))

final = agente_viagem.invoke({
    "messages": [HumanMessage(content=(
        "Planeje a viagem descrita no cenário. Escolha o destino, calcule o orçamento e produza um guia "
        "final com a recomendação, o resumo de custos e as ressalvas sobre as restrições. "
        "Grave o guia em /guia_final.md."
    ))],
    "todos": [], "files": {}, "cenario": CENARIO,
})

arquivos = final.get("files") or {}
verificar(T, "o agente produziu o guia final", "/guia_final.md" in arquivos, str(list(arquivos)))
verificar(T, "o agente delegou a escolha de destino", "/destinos.md" in arquivos)
verificar(T, "o agente delegou o orçamento", "/orcamento.md" in arquivos)
verificar(T, "o agente escreveu uma lista de tarefas", len(final.get("todos") or []) >= 3,
          str(final.get("todos")))
verificar(T, "todas as tarefas foram encerradas",
          all(t.get("status") == "concluida" for t in (final.get("todos") or [{}])),
          "verifique se o agente marcou o progresso")

tokens_ctx = contar_tokens_aproximado(final["messages"])
total_arquivos = sum(len(c) for c in arquivos.values())
verificar(T, "houve descarregamento de contexto", total_arquivos > tokens_ctx,
          f"contexto ~{tokens_ctx} tokens, arquivos {total_arquivos} caracteres")

resumo_verificacoes()

# 8. Checklist de entrega

Antes de entregar, verifique:

1. Todas as células foram executadas **na ordem**, e as saídas estão salvas no notebook.
2. A célula `resumo_verificacoes()` ao final da Tarefa 6 foi executada e a saída está visível.
3. O arquivo `skills/montar-dia-de-roteiro.md` está junto do notebook, assim como a configuração do
   servidor MCP usado na Tarefa 2 (comando/argumentos), se relevante para reproduzir sua execução.
4. As questões da análise crítica (Seção 7) estão respondidas.
5. As especificações escritas da Seção 8 (Plan-and-Execute e Reflection) estão preenchidas.
6. Nenhum valor do dicionário `CENARIO` foi alterado.
7. O arquivo segue o padrão de nome descrito na Seção 0.3 — Submissão.
